# V-JEPA 2 Demo Notebook

This tutorial provides an example of how to load the V-JEPA 2 model in vanilla PyTorch and HuggingFace, extract a video embedding, and then predict an action class. For more details about the paper and model weights, please see https://github.com/facebookresearch/vjepa2.

First, let's import the necessary libraries and load the necessary functions for this tutorial.

In [1]:
!pip install torch torchvision torchaudio
!pip install torchcodec

In [2]:
pip install -U git+https://github.com/huggingface/transformers


  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-889yndw7
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-889yndw7
  Resolved https://github.com/huggingface/transformers to commit c0fe6164d0d74e021aab9bfa0342a0f59ff59b16
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11327652 sha256=4ce253528da888b86e8ccd3baaec36e078ab2f962a3b0a4cf47ed44db8c54264
  Stored in directory: /tmp/pip-ephem-wheel-cache-8sgk9i29/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [5]:
import os # Moved import os to the top

# Clone the V-JEPA 2 repository if it doesn't exist
if not os.path.exists('vjepa2'):
    !git clone https://github.com/facebookresearch/vjepa2.git

# Add the cloned repository to Python's path
import sys
sys.path.insert(0, './vjepa2')

!pip install decord
import json
import subprocess

import numpy as np
import torch
import torch.nn.functional as F
from decord import VideoReader
from transformers import AutoVideoProcessor, AutoModel

import src.datasets.utils.video.transforms as video_transforms
import src.datasets.utils.video.volume_transforms as volume_transforms
from src.models.attentive_pooler import AttentiveClassifier
from src.models.vision_transformer import vit_giant_xformers_rope

IMAGENET_DEFAULT_MEAN = (0.485, 0.456, 0.406)
IMAGENET_DEFAULT_STD = (0.229, 0.224, 0.225)

def load_pretrained_vjepa_pt_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 encoder
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["encoder"]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    pretrained_dict = {k.replace("backbone.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def load_pretrained_vjepa_classifier_weights(model, pretrained_weights):
    # Load weights of the VJEPA2 classifier
    # The PyTorch state_dict is already preprocessed to have the right key names
    pretrained_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")["classifiers"][0]
    pretrained_dict = {k.replace("module.", ""): v for k, v in pretrained_dict.items()}
    msg = model.load_state_dict(pretrained_dict, strict=False)
    print("Pretrained weights found at {} and loaded with msg: {}".format(pretrained_weights, msg))


def build_pt_video_transform(img_size):
    short_side_size = int(256.0 / 224 * img_size)
    # Eval transform has no random cropping nor flip
    eval_transform = video_transforms.Compose(
        [
            video_transforms.Resize(short_side_size, interpolation="bilinear"),
            video_transforms.CenterCrop(size=(img_size, img_size)),
            volume_transforms.ClipToTensor(),
            video_transforms.Normalize(mean=IMAGENET_DEFAULT_MEAN, std=IMAGENET_DEFAULT_STD),
        ]
    )
    return eval_transform


def get_video():
    vr = VideoReader("sample_video.mp4")
    # choosing some frames here, you can define more complex sampling strategy
    frame_idx = np.arange(0, 128, 2)
    video = vr.get_batch(frame_idx).asnumpy()
    return video


def forward_vjepa_video(model_hf, model_pt, hf_transform, pt_transform):
    # Run a sample inference with VJEPA
    with torch.inference_mode():
        # Read and pre-process the image
        video = get_video()  # T x H x W x C
        video = torch.from_numpy(video).permute(0, 3, 1, 2)  # T x C x H x W
        x_pt = pt_transform(video).cuda().unsqueeze(0)
        x_hf = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")
        # Extract the patch-wise features from the last layer
        out_patch_features_pt = model_pt(x_pt)
        out_patch_features_hf = model_hf.get_vision_features(x_hf)

    return out_patch_features_hf, out_patch_features_pt


def get_vjepa_video_classification_results(classifier, out_patch_features_pt):
    SOMETHING_SOMETHING_V2_CLASSES = json.load(open("ssv2_classes.json", "r"))

    with torch.inference_mode():
        out_classifier = classifier(out_patch_features_pt)

    print(f"Classifier output shape: {out_classifier.shape}")

    print("Top 5 predicted class names:")
    top5_indices = out_classifier.topk(5).indices[0]
    top5_probs = F.softmax(out_classifier.topk(5).values[0]) * 100.0  # convert to percentage
    for idx, prob in zip(top5_indices, top5_probs):
        str_idx = str(idx.item())
        print(f"{SOMETHING_SOMETHING_V2_CLASSES[str_idx]} ({prob}%)")

    return

Cloning into 'vjepa2'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 316 (delta 140), reused 107 (delta 107), pack-reused 100 (from 1)
Receiving objects: 100% (316/316), 571.87 KiB | 13.30 MiB/s, done.
Resolving deltas: 100% (153/153), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 102.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Next, let's download a sample video to the local repository. If the video is already downloaded, the code will skip this step. Likewise, let's download a mapping for the action recognition classes used in Something-Something V2, so we can interpret the predicted action class from our model.

In [6]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.4/182.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.4 MB/s eta 0:00:00


Now, let's load the models in both vanilla Pytorch as well as through the HuggingFace API. Note that HuggingFace API will automatically load the weights through `from_pretrained()`, so there is no additional download required for HuggingFace.

To download the PyTorch model weights, use wget and specify your preferred target path. See the README for the model weight URLs.
E.g.
```
wget https://dl.fbaipublicfiles.com/vjepa2/vitg-384.pt -P YOUR_DIR
```
Then update `pt_model_path` with `YOUR_DIR/vitg-384.pt`. Also note that you have the option to use `torch.hub.load`.

In [13]:
!pip install -q yt-dlp

In [8]:
# HuggingFace model repo name
hf_model_name = (
    "facebook/vjepa2-vitg-fpc64-384"  # Replace with your favored model, e.g. facebook/vjepa2-vitg-fpc64-384
)
# Path to local PyTorch weights
#pt_model_path = "YOUR_MODEL_PATH"

# Initialize the HuggingFace model, load pretrained weights
model_hf = AutoModel.from_pretrained(hf_model_name)
model_hf.cuda().eval()

# Build HuggingFace preprocessing transform
hf_transform = AutoVideoProcessor.from_pretrained(hf_model_name)
img_size = hf_transform.crop_size["height"]  # E.g. 384, 256, etc.

# Initialize the PyTorch model, load pretrained weights
#model_pt = vit_giant_xformers_rope(img_size=(img_size, img_size), num_frames=64)
#model_pt.cuda().eval()
#load_pretrained_vjepa_pt_weights(model_pt, pt_model_path)

### Can also use torch.hub to load the model
# model_pt, _ = torch.hub.load('facebookresearch/vjepa2', 'vjepa2_vit_giant_384')
# model_pt.cuda().eval()

# Build PyTorch preprocessing transform
#pt_video_transform = build_pt_video_transform(img_size=img_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/780 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

Now we can run the encoder on the video to get the patch-wise features from the last layer of the encoder. To verify that the HuggingFace and PyTorch models are equivalent, we will compare the values of the features.

In [9]:
import json
from pathlib import Path

COIN_JSON_PATH = "/content/COIN.json"   # change this if needed

with open(COIN_JSON_PATH, "r") as f:
    coin_data = json.load(f)["database"]

print("num videos in COIN annotations:", len(coin_data))

# show one example key
first_vid = next(iter(coin_data))
print("example video id:", first_vid)
print("example fields:", coin_data[first_vid].keys())

num videos in COIN annotations: 11827
example video id: xZecGPPhbHE
example fields: dict_keys(['recipe_type', 'annotation', 'video_url', 'start', 'end', 'duration', 'class', 'subset'])


In [37]:
import json

COIN_JSON_PATH = "COIN.json"

with open(COIN_JSON_PATH, "r") as f:
    coin_data = json.load(f)["database"]

print("Number of videos:", len(coin_data))

first_video_id = next(iter(coin_data))
print("Example video id:", first_video_id)
print("Example entry keys:", coin_data[first_video_id].keys())
print("Example class:", coin_data[first_video_id]["class"])
print("Example number of segments:", len(coin_data[first_video_id]["annotation"]))

Number of videos: 11827
Example video id: xZecGPPhbHE
Example entry keys: dict_keys(['recipe_type', 'annotation', 'video_url', 'start', 'end', 'duration', 'class', 'subset'])
Example class: PutOnHairExtensions
Example number of segments: 3


In [59]:
def select_coin_videos(coin_data, subset="training", max_videos=100):
    selected = []

    for video_id, entry in coin_data.items():
        if entry.get("subset") != subset:
            continue

        if len(entry.get("annotation", [])) < 2:
            continue

        selected.append((video_id, entry))

        if len(selected) >= max_videos:
            break

    return selected


selected_videos = select_coin_videos(coin_data, subset="training", max_videos=100)

print("Selected videos:")
for video_id, entry in selected_videos:
    print(video_id, "|", entry["class"], "| segments:", len(entry["annotation"]))

Selected videos:
NLy71UrHElw | PractisePoleVault | segments: 3
JTmzWWU7s1M | MakeSoap | segments: 2
jXYZEb-56Wc | PasteScreenProtectorOnPad | segments: 5
ar2vXJj_dQI | ReplaceFaucet | segments: 3
oDAY5PMwZgU | UseVendingMachine | segments: 3
r_5RtkDwNsg | MakeSalad | segments: 2
CWmC03KVuPU | MakeTea | segments: 7
ErhvSbJZ6mg | DoLinoPrinting | segments: 6
DvaCfRWskRE | PackSleepingBag | segments: 2
ot_sBwstblw | PractisePoleVault | segments: 3
gnlUBK-cvfc | MakeBurger | segments: 3
n0HdUsT48p8 | PerformPaperToMoneyTrick | segments: 3
v4z4s8a7YqE | TieBoatToDock | segments: 3
iD9JmUkGpOA | CleanShrimp | segments: 4
SN_IhH4PDPo | RefillALighter | segments: 3
cK55Meyf3ss | InstallShowerHead | segments: 3
WjjexIlQg6k | BoilNoodles | segments: 4
EWrpjhOtuss | MakeSalmon | segments: 4
8TVl3Ui-WHg | PumpUpBicycleTire | segments: 3
WlV0MTSuzOk | ReplaceFilterForAirPurifier | segments: 5
LMoGwLqPM64 | BoilNoodles | segments: 3
1lip9rOA79c | PrepareStandardSolution | segments: 4
U-s6EYzqjmA | U

In [60]:
from pathlib import Path

LATENT_DIR = Path("coin_latents")
LATENT_DIR.mkdir(exist_ok=True)

TMP_VIDEO_PATH = Path("tmp_coin_video.mp4")

MANIFEST_PATH = LATENT_DIR / "manifest.jsonl"

print("Latent dir:", LATENT_DIR)
print("Temp video path:", TMP_VIDEO_PATH)

Latent dir: coin_latents
Temp video path: tmp_coin_video.mp4


In [61]:
import subprocess

def download_coin_video(video_url, out_path):

    if out_path.exists():
        out_path.unlink()

    cmd = [
        "yt-dlp",
        "-f", "best[ext=mp4]",
        "--merge-output-format", "mp4",
        "-o", str(out_path),
        video_url
    ]

    try:
        subprocess.run(cmd, check=True)
        print("Download successful:", out_path)
        return True

    except subprocess.CalledProcessError:
        print("Failed to download:", video_url)
        return False

In [62]:
import numpy as np
import torch
from decord import VideoReader

def sample_frames_from_time_window(vr, start_sec, end_sec, T=32):

    fps = vr.get_avg_fps()
    total_frames = len(vr)

    start_f = max(0, int(start_sec * fps))
    end_f = min(total_frames - 1, int(end_sec * fps))

    if end_f <= start_f:
        end_f = min(total_frames - 1, start_f + 1)

    idx = np.linspace(start_f, end_f, T).astype(int)

    video = vr.get_batch(idx).asnumpy()
    video = torch.from_numpy(video).permute(0,3,1,2)

    return video

In [63]:
import gc

def extract_segment_latent_from_video_reader(vr, start_sec, end_sec, model_hf, hf_transform, T=32):

    video = sample_frames_from_time_window(vr, start_sec, end_sec, T=T)

    x = hf_transform(video, return_tensors="pt")["pixel_values_videos"].to("cuda")

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            tokens = model_hf.get_vision_features(x)
            z = tokens.mean(dim=1)

    z_cpu = z.detach().cpu()

    del video, x, tokens, z
    torch.cuda.empty_cache()
    gc.collect()

    return z_cpu.squeeze(0)

In [66]:
for video_id, entry in selected_videos:

    tensor_path = LATENT_DIR / f"{video_id}.pt"

    if tensor_path.exists():
        print("Already processed:", video_id)
        continue

    print("\nProcessing video:", video_id, entry["class"])

    video_url = entry["video_url"]

    ok = download_coin_video(video_url, TMP_VIDEO_PATH)

    if not ok:
        print("Skipping video due to download failure")
        continue

    try:
      vr = VideoReader(str(TMP_VIDEO_PATH))
    except Exception as e:
      print(f"Failed to read video with Decord: {video_id}")
      print("Reason:", e)
      if TMP_VIDEO_PATH.exists():
          TMP_VIDEO_PATH.unlink()
          print("Deleted temp video after read failure")
      continue

    z_segments = []
    segment_meta = []

    for seg_idx, ann in enumerate(entry["annotation"]):

        start_sec, end_sec = ann["segment"]

        z_t = extract_segment_latent_from_video_reader(
            vr,
            start_sec,
            end_sec,
            model_hf,
            hf_transform,
            T=32
        )

        z_segments.append(z_t)

        segment_meta.append({
            "segment_index": seg_idx,
            "step_id": ann["id"],
            "step_label": ann["label"],
            "start_sec": start_sec,
            "end_sec": end_sec
        })

        print(f"Segment {seg_idx+1}/{len(entry['annotation'])} done")

    z_segments = torch.stack(z_segments)

    delta_z = z_segments[1:] - z_segments[:-1]

    torch.save(
        {
            "video_id": video_id,
            "z_segments": z_segments,
            "delta_z": delta_z,
            "segment_meta": segment_meta
        },
        tensor_path
    )

    print("Saved latent file:", tensor_path)

    TMP_VIDEO_PATH.unlink()

    print("Deleted temp video")

Already processed: NLy71UrHElw
Already processed: JTmzWWU7s1M
Already processed: jXYZEb-56Wc
Already processed: ar2vXJj_dQI
Already processed: oDAY5PMwZgU
Already processed: r_5RtkDwNsg
Already processed: CWmC03KVuPU

Processing video: ErhvSbJZ6mg DoLinoPrinting
Failed to download: https://www.youtube.com/embed/ErhvSbJZ6mg
Skipping video due to download failure

Processing video: DvaCfRWskRE PackSleepingBag
Failed to download: https://www.youtube.com/embed/DvaCfRWskRE
Skipping video due to download failure

Processing video: ot_sBwstblw PractisePoleVault
Failed to download: https://www.youtube.com/embed/ot_sBwstblw
Skipping video due to download failure
Already processed: gnlUBK-cvfc
Already processed: n0HdUsT48p8
Already processed: v4z4s8a7YqE

Processing video: iD9JmUkGpOA CleanShrimp
Failed to download: https://www.youtube.com/embed/iD9JmUkGpOA
Skipping video due to download failure
Already processed: SN_IhH4PDPo
Already processed: cK55Meyf3ss
Already processed: WjjexIlQg6k

Process

In [54]:
data = torch.load("coin_latents/NLy71UrHElw.pt")

for i, meta in enumerate(data["segment_meta"]):
    print(f"Segment {i}: {meta['step_label']}")
    print("z_t first 10 values:", data["z_segments"][i][:10])
    print()

Segment 0: begin to run up
z_t first 10 values: tensor([ 1.1484, -0.8192,  0.2657, -0.7826,  0.1668,  0.7971,  0.0642,  0.5980,
         0.4075,  0.1378])

Segment 1: begin to jump up
z_t first 10 values: tensor([ 5.7843e-01, -1.2593e+00, -1.8211e-01, -4.8593e-01,  7.3142e-01,
        -3.5069e-02, -9.7111e-02,  1.7575e-01,  6.9465e-04, -7.2796e-01])

Segment 2: fall to the ground
z_t first 10 values: tensor([ 0.4515, -0.8440, -0.1669, -0.5265,  0.3367,  0.2071,  0.0987,  0.2320,
        -0.1074, -0.4380])



In [55]:
for i in range(len(data["delta_z"])):
    print(f"Delta z {i}: {data['segment_meta'][i]['step_label']} -> {data['segment_meta'][i+1]['step_label']}")
    print("delta_z first 10 values:", data["delta_z"][i][:10])
    print()

Delta z 0: begin to run up -> begin to jump up
delta_z first 10 values: tensor([-0.5700, -0.4401, -0.4478,  0.2967,  0.5646, -0.8322, -0.1613, -0.4223,
        -0.4068, -0.8658])

Delta z 1: begin to jump up -> fall to the ground
delta_z first 10 values: tensor([-0.1270,  0.4153,  0.0152, -0.0406, -0.3947,  0.2422,  0.1958,  0.0563,
        -0.1081,  0.2899])

